In [ ]:
!pip uninstall -y transformers peft accelerate -q

!pip install -q \
    transformers==4.41.2 \
    peft==0.11.1 \
    accelerate==0.30.1 \
    datasets \
    sentencepiece \
    safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
output_dir = "/content/drive/MyDrive/CLIP_Adapter_MirageNews"
os.makedirs(output_dir, exist_ok=True)
print("Save to:", output_dir)

Mounted at /content/drive
Save to: /content/drive/MyDrive/CLIP_Adapter_MirageNews


In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

dataset = load_dataset("anson-huang/mirage-news")
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name).to(device)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [ ]:
def preprocess(examples):
    inputs = processor(
        text=examples["text"],
        images=examples["image"],
        padding="max_length",
        truncation=True,
        max_length=77
    )
    inputs["labels"] = examples["label"]
    return inputs

encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names,
    batched=True
)

encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from torch import nn

class Adapter(nn.Module):
    def __init__(self, hidden_dim, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(hidden_dim, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, hidden_dim)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

class CLIPAdapterForMFND(nn.Module):
    def __init__(self, clip_model, num_labels=2, bottleneck=64):
        super().__init__()
        self.clip = clip_model

        # Freeze backbone
        for p in self.clip.parameters():
            p.requires_grad = False

        text_hidden = clip_model.config.text_config.hidden_size
        vision_hidden = clip_model.config.vision_config.hidden_size

        self.text_adapter = Adapter(text_hidden, bottleneck)
        self.vision_adapter = Adapter(vision_hidden, bottleneck)

        self.classifier = nn.Linear(
            clip_model.config.projection_dim * 2,
            num_labels
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        text_outputs = self.clip.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_hidden = self.text_adapter(text_outputs.last_hidden_state)
        text_cls = text_hidden[:, 0, :]
        text_features = self.clip.text_projection(text_cls)

        vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
        vision_hidden = self.vision_adapter(vision_outputs.last_hidden_state)
        image_cls = vision_hidden[:, 0, :]
        image_features = self.clip.visual_projection(image_cls)

        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        fused = torch.cat([text_features, image_features], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = CLIPAdapterForMFND(clip_model).to(device)

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Trainable params: 167,298
Total params: 151,444,611
Trainable %: 0.1105%


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()
metrics = trainer.evaluate()
print(metrics)

trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("✅ Model & processor saved to:", output_dir)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.273000,0.248415,0.936800,0.936102,0.937600,0.936851,0.984292
2,0.150100,0.158507,0.953200,0.954290,0.952000,0.953144,0.990987
3,0.110900,0.130060,0.957200,0.957566,0.956800,0.957183,0.993170
4,0.096400,0.119479,0.956800,0.952456,0.961600,0.957006,0.993933
5,0.098100,0.116096,0.958800,0.958433,0.959200,0.958816,0.994102


{'eval_loss': 0.11609572917222977, 'eval_accuracy': 0.9588, 'eval_precision': 0.9584332533972821, 'eval_recall': 0.9592, 'eval_f1': 0.9588164734106357, 'eval_auc': 0.99410208, 'eval_runtime': 39.6121, 'eval_samples_per_second': 63.112, 'eval_steps_per_second': 3.963, 'epoch': 5.0}
✅ Model & processor saved to: /content/drive/MyDrive/CLIP_Adapter_MirageNews
